In [1]:
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
# Light-level readings
light_values = np.array([
    100, 300, 500, 700,
    1000, 1300, 1600, 2000,
    2500, 3000, 3500, 4000
], dtype=np.float32)

# Labels
# 0 = Dark
# 1 = Normal
# 2 = Bright

labels = np.array([
    0, 0, 0, 0,
    1, 1, 1, 1,
    2, 2, 2, 2
], dtype=np.int32)

print("Light readings:")
print(light_values)

print("\nLabels:")
print(labels)

Light readings:
[ 100.  300.  500.  700. 1000. 1300. 1600. 2000. 2500. 3000. 3500. 4000.]

Labels:
[0 0 0 0 1 1 1 1 2 2 2 2]


In [3]:
# ESP32/Wokwi-style ADC range: 0–4095
x_train = light_values / 4095.0

print("Normalized data:")
print(x_train)

Normalized data:
[0.02442002 0.07326008 0.12210012 0.17094018 0.24420024 0.31746033
 0.3907204  0.4884005  0.61050063 0.73260075 0.85470086 0.976801  ]


In [4]:
class_names = [
    "Dark",
    "Normal",
    "Bright"
]

for i, name in enumerate(class_names):
    print(i, "=", name)

0 = Dark
1 = Normal
2 = Bright


In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 195 (780.00 B)

 Trainable params: 195 (780.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [7]:
history = model.fit(
    x_train,
    labels,
    epochs=200,
    verbose=0
)

print("Model training completed.")

Model training completed.


In [8]:
test_values = np.array([
    400,
    1700,
    3300
], dtype=np.float32)

test_input = test_values / 4095.0

predictions = model.predict(
    test_input,
    verbose=0
)

print("Original TensorFlow Model Predictions")
print("--------------------------------------")

original_classes = []

for value, prediction in zip(test_values, predictions):

    predicted_class = np.argmax(prediction)

    original_classes.append(predicted_class)

    print(
        f"Light Reading: {value:.0f} | "
        f"Prediction: {class_names[predicted_class]}"
    )

Original TensorFlow Model Predictions
--------------------------------------
Light Reading: 400 | Prediction: Normal
Light Reading: 1700 | Prediction: Normal
Light Reading: 3300 | Prediction: Bright


In [9]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("light_classifier.tflite", "wb") as file:
    file.write(tflite_model)

print("TensorFlow Lite conversion completed!")
print("Created file: light_classifier.tflite")

Saved artifact at '/tmp/tmpdjqortja'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132075964019536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132075964020496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132075964020112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132075964020880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132075964020688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132075964021264: TensorSpec(shape=(), dtype=tf.resource, name=None)
TensorFlow Lite conversion completed!
Created file: light_classifier.tflite


In [10]:
import os

file_size = os.path.getsize("light_classifier.tflite")

print("TFLite model created successfully.")
print("File size:", file_size, "bytes")

TFLite model created successfully.
File size: 2880 bytes


In [11]:
interpreter = tf.lite.Interpreter(
    model_path="light_classifier.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite model loaded successfully.")

print("\nInput details:")
print(input_details)

print("\nOutput details:")
print(output_details)

TFLite model loaded successfully.

Input details:
[{'name': 'serving_default_keras_tensor:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]

Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 10, 'shape': array([1, 3], dtype=int32), 'shape_signature': array([-1,  3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [12]:
tflite_classes = []

print("TFLite Model Predictions")
print("------------------------")

for value in test_values:

    input_data = np.array(
        [[value / 4095.0]],
        dtype=np.float32
    )

    interpreter.set_tensor(
        input_details[0]["index"],
        input_data
    )

    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    predicted_class = np.argmax(output)

    tflite_classes.append(predicted_class)

    print(
        f"Light Reading: {value:.0f} | "
        f"Prediction: {class_names[predicted_class]}"
    )

TFLite Model Predictions
------------------------
Light Reading: 400 | Prediction: Normal
Light Reading: 1700 | Prediction: Normal
Light Reading: 3300 | Prediction: Bright


In [13]:
print("Inference Comparison")
print("--------------------")

for value, original, tflite in zip(
    test_values,
    original_classes,
    tflite_classes
):

    print(
        f"Light: {value:.0f} | "
        f"TensorFlow: {class_names[original]} | "
        f"TFLite: {class_names[tflite]}"
    )

if original_classes == tflite_classes:
    print("\nInference parity: PASS")
else:
    print("\nInference parity: FAIL")

Inference Comparison
--------------------
Light: 400 | TensorFlow: Normal | TFLite: Normal
Light: 1700 | TensorFlow: Normal | TFLite: Normal
Light: 3300 | TensorFlow: Bright | TFLite: Bright

Inference parity: PASS


In [14]:
print("=" * 45)
print("DAY 12 – TENSORFLOW LITE CONVERSION")
print("=" * 45)

print("\nModel: Light Level Classifier")
print("Classes: Dark / Normal / Bright")

print("\nTest Results:")

for value, original, tflite in zip(
    test_values,
    original_classes,
    tflite_classes
):
    print(
        f"{value:.0f} → "
        f"{class_names[tflite]}"
    )

print("\nTFLite file: light_classifier.tflite")

if original_classes == tflite_classes:
    print("Inference parity: PASS")
else:
    print("Inference parity: FAIL")

DAY 12 – TENSORFLOW LITE CONVERSION

Model: Light Level Classifier
Classes: Dark / Normal / Bright

Test Results:
400 → Normal
1700 → Normal
3300 → Bright

TFLite file: light_classifier.tflite
Inference parity: PASS
